In [0]:
-- Fetch the qdp_transactions_all document data product created by 01. Quantexa Processing of Raw Data in Bronze

-- 01-1. Create a point in time run_timestamp for all data inserts
DROP TEMPORARY VARIABLE IF EXISTS run_timestamp;

DECLARE run_timestamp TIMESTAMP DEFAULT current_timestamp();

SELECT run_timestamp;


-- 01-2. Create default start date for all data inserts

DROP TEMPORARY VARIABLE IF EXISTS default_start_date;

DECLARE default_start_date STRING;

SET  VAR default_start_date = '01-01-1900';

SELECT default_start_date;


-- 01-3. Create default end date for all data inserts
DROP TEMPORARY VARIABLE IF EXISTS default_end_date;

DECLARE default_end_date STRING;

SET  VAR default_end_date = '31-12-2999';

SELECT default_end_date;


 
-- 2. Remove all Account Transaction records for the Tennant

DELETE FROM demo_banking_silver.qdp_transactions_all.sat_transaction s
WHERE s.transaction_id IN (
  SELECT DISTINCT transaction_id
  FROM demo_banking_silver.qdp_transactions_all.hub_transaction
  WHERE tennant_id = :p_tennant_id)
;

DELETE FROM demo_banking_silver.qdp_transactions_all.hub_transaction WHERE tennant_id = :p_tennant_id;


-- 4. Assign the correct bene_account_id or leave as defaulted zero value if does not exist

CREATE OR REPLACE TEMP VIEW account_transactions_correctly_typed_baccid AS
SELECT 
    q.*,
    COALESCE(acc.account_id, 0) AS bene_account_id
FROM demo_banking_silver.qdp_from_quantexa_banking_supply_chain.qdp_transactions_all q
LEFT JOIN demo_banking_silver.qdp_accounts_all.hub_account acc
ON acc.account_entity_id = q.bene_account_entity_id
;


SELECT * from account_transactions_correctly_typed_baccid;


-- 5. Assign the correct orig_account_id or leave as defaulted zero value if does not exist
CREATE OR REPLACE TEMP VIEW account_transactions_correctly_typed_baccid_oaccid AS
SELECT 
    q.*,
    COALESCE(acc.account_id, 0) AS orig_account_id
FROM account_transactions_correctly_typed_baccid q
LEFT JOIN demo_banking_silver.qdp_accounts_all.hub_account acc
ON acc.account_entity_id = q.orig_account_entity_id
;


SELECT * from account_transactions_correctly_typed_baccid_oaccid;



-- 7. Create the  hub_transactions table records
INSERT INTO demo_banking_silver.qdp_transactions_all.hub_transaction
(bene_account_entity_id, 
tennant_id,
transaction_entity_id, 
orig_account_entity_id,
bene_account_id, 
orig_account_id,
period_start, 
period_end)
SELECT 
  trans.bene_account_entity_id,
  trans.tennant_id, 
  trans.transaction_entity_id, 
  trans.orig_account_entity_id, 
  trans.bene_account_id, 
  trans.orig_account_id, 
  trans.date, 
  trans.date
FROM account_transactions_correctly_typed_baccid_oaccid trans  
;


SELECT * FROM demo_banking_silver.qdp_transactions_all.hub_transaction;


-- 8. Create the sat_transactions_all table records
INSERT INTO demo_banking_silver.qdp_transactions_all.sat_transaction
    ( 
     transaction_id, 
     load_datetime, 
     record_source_id,
--
     amount,
     datetime,
     debit_or_credit_code,
     type_code,
     original_account_number,
     original_account_name,
     original_country_code,
     account_number,
     beneficiary_name,
     country_code,
     description,
     data_source_code,
     original_bank,
     original_bank_country_code,
     institution_bank,
     institution_bank_country_code,
     sending_bank,
     sending_bank_country_code,
     beneficiary_bank,
     beneficiary_bank_country_code,
     original_currency_code,
     currency_code,
--

     period_start,
     period_end 
    )
SELECT
  h.transaction_id AS transaction_id,
  run_timestamp AS load_datetime,
  try_cast(0 AS BIGINT) AS record_source_id,
--
  amount_usd AS amount,
  q.date,
  code AS debit_or_credit_code,
  txn_type AS type_code,
  orig_account_no AS original_account_number,
  orig_name AS original_account_name,
  orig_country AS original_country_code,
  bene_account_no AS account_number,
  beneficiary_name AS beneficiary_name,
  bene_country AS country_code,
  description AS description,
  source_system AS data_source_code,
  orig_bank AS original_bank,
  orig_bank_country AS original_bank_country_code,
  institution_bank AS institution_bank,
  institution_bank_country AS institution_bank_country_code,
  sending_bank AS sending_bank,
  sending_bank_country AS sending_bank_country_code,
  bene_bank AS beneficiary_bank,
  bene_bank_country AS beneficiary_bank_country_code,
  orig_currency AS original_currency_code,
  base_ccy AS currency_code,
--
  q.date AS period_start,
  q.date AS period_end

FROM account_transactions_correctly_typed_baccid_oaccid q
JOIN  demo_banking_silver.qdp_transactions_all.hub_transaction h
  ON h.transaction_entity_id = q.transaction_entity_id
;
  
SELECT * FROM  demo_banking_silver.qdp_transactions_all.sat_transaction;

